# QuantumDLBlocks: Quantum-Geometric Neural Architecture Search (Q-NAS)

This notebook compares **all three Quantum-Geometric NAS paradigms** head-to-head against Classical Softmax DARTS:
1. **Coherent Superposition DARTS (`SuperpositionDARTSNetwork`)**: Places candidate quantum operations (`spherical_skip`, `born_attn_real`, `born_attn_complex`, `projective_mlp`, `sparse_born_moe`) in a coherent quantum superposition and anneals `PowerPurification` ($\rho^p / \mathrm{Tr}(\rho^p)$) from $p=1 \to 4$ to smoothly collapse to a pure discrete architecture without the DARTS discretization gap.
2. **Thermodynamic / von Neumann Entropy-Adaptive Kraus Network (`EntropyAdaptiveNetwork`)**: Monitors layerwise von Neumann entropy $S(\rho_l) = -\mathrm{Tr}(\rho_l \ln \rho_l)$ and purity $\gamma(\rho_l) = \mathrm{Tr}(\rho_l^2)$ during training to dynamically expand saturated Hilbert subspaces, compress low-entropy eigenspaces, or activate polynomial purification.
3. **Zero-Shot Quantum Proxy Search (`QuantumZeroShotNAS`)**: Evaluates candidate architectures at initialization using a single minibatch forward/backward pass combining Hilbert-Space Effective Rank, Stiefel SynFlow Gradient Isotropy, Born Attention Entropy, and Stiefel Isometry Retention.

In [ ]:
# Install QuantumDLBlocks (if running on Google Colab)
# !pip install -q git+https://github.com/festivity/QuantumDLBlocks.git scikit-learn
import torch
from qdlblocks.qnas import (
    EntropyAdaptiveNetwork,
    QuantumZeroShotNAS,
    SuperpositionDARTSNetwork,
    load_qnas_dataset,
    run_qnas_comparison,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running Q-NAS comparison on: {device}")

## 1. Execute Head-to-Head Comparison Across All Q-NAS Paradigms

In [ ]:
report = run_qnas_comparison(
    device=device,
    epochs=20,
    num_zero_shot_candidates=12,
    seed=42,
)
print(report["markdown_table"])
print("\nZero-Shot Proxy Rank Correlations:", report["proxy_rank_correlations"])

## 2. Inspect Dynamic Hilbert Space Adaptation Trajectory (`EntropyAdaptiveNetwork`)

In [ ]:
x_tr, y_tr, x_te, y_te = load_qnas_dataset(seed=42)
x_tr, y_tr = x_tr.to(device), y_tr.to(device)

net = EntropyAdaptiveNetwork(
    in_features=64,
    stem_dim=8,
    init_dims=(8, 8, 8),
    num_classes=10,
    expand_entropy_ratio=0.55,
).to(device)

for step in range(3):
    rec = net.step_topology_adaptation(x_tr[:128])
    print(f"Adaptation Step {step+1}: dims={rec['layer_dims']}, powers={rec['purification_powers']}, events={rec['events']}")